# UAV-VisLoc Benchmark

Runs all feature-matching pipelines on the local UAV-VisLoc dataset.

**Dataset:** `UAV_VisLoc_example/03`  
**Limit:** 100 images per run (change `--limit` in each cell)  
**Success threshold:** 25 m GPS error  
**Results & visualizations:** saved to `results/` next to this notebook

Each section is independent — cells can be run in any order.

In [4]:
import sys, os

REPO = os.path.dirname(os.path.abspath('__file__'))
DATA = os.path.join(REPO, 'UAV_VisLoc_example')
OUT  = os.path.join(REPO, 'results')
os.makedirs(OUT, exist_ok=True)

if REPO not in sys.path:
    sys.path.insert(0, REPO)

print(f'REPO: {REPO}')
print(f'DATA: {DATA}')
print(f'OUT:  {OUT}')

REPO: /kaggle/working
DATA: /kaggle/working/UAV_VisLoc_example
OUT:  /kaggle/working/results


---
## Section 1 — Baseline (OpenCV)

Classical feature detectors using OpenCV. No GPU required, no extra installs.

| Method | Detector | Descriptor | Matcher |
|--------|----------|------------|---------|
| SIFT   | DoG keypoints | 128-d float SIFT | FLANN + Lowe ratio 0.75 |
| ORB    | FAST keypoints | 256-bit binary ORB | BFMatcher Hamming |
| BRISK  | AGAST keypoints | 512-bit binary BRISK | BFMatcher Hamming |

### 1a — Baseline SIFT

Scale-Invariant Feature Transform. Most accurate of the three classical methods. Slower than ORB/BRISK.

In [5]:
import sys, os
REPO = os.path.dirname(os.path.abspath('__file__'))
DATA = os.path.join(REPO, 'UAV_VisLoc_example')
OUT  = os.path.join(REPO, 'results')
os.makedirs(OUT, exist_ok=True)
sys.modules.pop('Baseline_pipeline', None)
sys.modules.pop('visloc_utils', None)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import Baseline_pipeline as pl

pl.BASE      = f"{DATA}/03"
pl.SAT_TIF   = f"{pl.BASE}/satellite03.tif"
pl.DRONE_DIR = f"{pl.BASE}/drone"
pl.DRONE_CSV = f"{pl.BASE}/03.csv"
pl.SAT_CSV   = f"{DATA}/satellite_ coordinates_range.csv"
pl.OUT_CSV   = f"{OUT}/baseline_sift_results.csv"
pl.VIZ_DIR   = f"{OUT}/baseline_sift_viz"

sys.argv = ['', '--limit', '100', '--method', 'sift', '--dist', '25', '--visualize']
pl.main()

ModuleNotFoundError: No module named 'Baseline_pipeline'

### 1b — Baseline ORB

Oriented FAST and Rotated BRIEF. Very fast binary descriptor. Less accurate than SIFT but runs in real time.

In [ ]:
import sys, os
REPO = os.path.dirname(os.path.abspath('__file__'))
DATA = os.path.join(REPO, 'UAV_VisLoc_example')
OUT  = os.path.join(REPO, 'results')
os.makedirs(OUT, exist_ok=True)
sys.modules.pop('Baseline_pipeline', None)
sys.modules.pop('visloc_utils', None)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import Baseline_pipeline as pl

pl.BASE      = f"{DATA}/03"
pl.SAT_TIF   = f"{pl.BASE}/satellite03.tif"
pl.DRONE_DIR = f"{pl.BASE}/drone"
pl.DRONE_CSV = f"{pl.BASE}/03.csv"
pl.SAT_CSV   = f"{DATA}/satellite_ coordinates_range.csv"
pl.OUT_CSV   = f"{OUT}/baseline_orb_results.csv"
pl.VIZ_DIR   = f"{OUT}/baseline_orb_viz"

sys.argv = ['', '--limit', '100', '--method', 'orb', '--dist', '25', '--visualize']
pl.main()

### 1c — Baseline BRISK

Binary Robust Invariant Scalable Keypoints. Similar speed to ORB with a larger 512-bit descriptor. Often more robust to scale changes than ORB.

In [ ]:
import sys, os
REPO = os.path.dirname(os.path.abspath('__file__'))
DATA = os.path.join(REPO, 'UAV_VisLoc_example')
OUT  = os.path.join(REPO, 'results')
os.makedirs(OUT, exist_ok=True)
sys.modules.pop('Baseline_pipeline', None)
sys.modules.pop('visloc_utils', None)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import Baseline_pipeline as pl

pl.BASE      = f"{DATA}/03"
pl.SAT_TIF   = f"{pl.BASE}/satellite03.tif"
pl.DRONE_DIR = f"{pl.BASE}/drone"
pl.DRONE_CSV = f"{pl.BASE}/03.csv"
pl.SAT_CSV   = f"{DATA}/satellite_ coordinates_range.csv"
pl.OUT_CSV   = f"{OUT}/baseline_brisk_results.csv"
pl.VIZ_DIR   = f"{OUT}/baseline_brisk_viz"

sys.argv = ['', '--limit', '100', '--method', 'brisk', '--dist', '25', '--visualize']
pl.main()

---
## Section 2 — LightGlue

Learned keypoint matcher that works with multiple front-end detectors. Uses an attention-based GNN to prune ambiguous matches. GPU strongly recommended.

| Variant | Detector | Descriptor | Notes |
|---------|----------|------------|-------|
| DISK    | DISK (learned) | DISK (256-d) | Best accuracy |
| SIFT    | DoG | SIFT (128-d) | Good balance |
| DeDoDe-B | DeDoDe (learned) | DeDoDe (256-d) | Strongest detector |

### 2a — LightGlue + DISK

DISK detector with LightGlue matcher. Typically the strongest LightGlue variant for outdoor aerial imagery.

In [6]:
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', 'lightglue', '-q'], check=True)

REPO = os.path.dirname(os.path.abspath('__file__'))
DATA = os.path.join(REPO, 'UAV_VisLoc_example')
OUT  = os.path.join(REPO, 'results')
os.makedirs(OUT, exist_ok=True)
sys.modules.pop('lightglue_pipeline', None)
sys.modules.pop('visloc_utils', None)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import lightglue_pipeline as pl

pl.BASE      = f"{DATA}/03"
pl.SAT_TIF   = f"{pl.BASE}/satellite03.tif"
pl.DRONE_DIR = f"{pl.BASE}/drone"
pl.DRONE_CSV = f"{pl.BASE}/03.csv"
pl.SAT_CSV   = f"{DATA}/satellite_ coordinates_range.csv"
pl.OUT_CSV   = f"{OUT}/lightglue_disk_results.csv"
pl.VIZ_DIR   = f"{OUT}/lightglue_disk_viz"

sys.argv = ['', '--limit', '100', '--method', 'disk',
            '--ransac-thresh', '10', '--min-inl', '6', '--visualize']
pl.main()

ERROR: Could not find a version that satisfies the requirement lightglue (from versions: none)
ERROR: No matching distribution found for lightglue


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', 'lightglue', '-q']' returned non-zero exit status 1.

### 2b — LightGlue + SIFT

Classic SIFT detector fed into the LightGlue matcher. Good fallback when DISK/DeDoDe weights are unavailable or slow to download.

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', 'lightglue', '-q'], check=True)

REPO = os.path.dirname(os.path.abspath('__file__'))
DATA = os.path.join(REPO, 'UAV_VisLoc_example')
OUT  = os.path.join(REPO, 'results')
os.makedirs(OUT, exist_ok=True)
sys.modules.pop('lightglue_pipeline', None)
sys.modules.pop('visloc_utils', None)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import lightglue_pipeline as pl

pl.BASE      = f"{DATA}/03"
pl.SAT_TIF   = f"{pl.BASE}/satellite03.tif"
pl.DRONE_DIR = f"{pl.BASE}/drone"
pl.DRONE_CSV = f"{pl.BASE}/03.csv"
pl.SAT_CSV   = f"{DATA}/satellite_ coordinates_range.csv"
pl.OUT_CSV   = f"{OUT}/lightglue_sift_results.csv"
pl.VIZ_DIR   = f"{OUT}/lightglue_sift_viz"

sys.argv = ['', '--limit', '100', '--method', 'sift',
            '--ransac-thresh', '10', '--min-inl', '6', '--visualize']
pl.main()

### 2c — LightGlue + DeDoDe-B

DeDoDe detector (trained to detect repeatable keypoints across viewpoints) with LightGlue. Often finds more matches in low-texture regions than SIFT or DISK.

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', 'lightglue', '-q'], check=True)

REPO = os.path.dirname(os.path.abspath('__file__'))
DATA = os.path.join(REPO, 'UAV_VisLoc_example')
OUT  = os.path.join(REPO, 'results')
os.makedirs(OUT, exist_ok=True)
sys.modules.pop('lightglue_pipeline', None)
sys.modules.pop('visloc_utils', None)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import lightglue_pipeline as pl

pl.BASE      = f"{DATA}/03"
pl.SAT_TIF   = f"{pl.BASE}/satellite03.tif"
pl.DRONE_DIR = f"{pl.BASE}/drone"
pl.DRONE_CSV = f"{pl.BASE}/03.csv"
pl.SAT_CSV   = f"{DATA}/satellite_ coordinates_range.csv"
pl.OUT_CSV   = f"{OUT}/lightglue_dedodeb_results.csv"
pl.VIZ_DIR   = f"{OUT}/lightglue_dedodeb_viz"

sys.argv = ['', '--limit', '100', '--method', 'dedodeb',
            '--ransac-thresh', '10', '--min-inl', '6', '--visualize']
pl.main()

---
## Section 3 — LoFTR

**LoFTR** (Detector-Free Local Feature Matching with Transformers) matches dense pixel pairs directly without detecting keypoints first. Uses a coarse-to-fine Transformer architecture trained on MegaDepth (`outdoor` weights). Strong in low-texture and repetitive regions where keypoint detectors struggle. GPU required for reasonable speed.

### 3a — LoFTR (outdoor)

Pretrained on MegaDepth outdoor scenes. Correct weight set for UAV/satellite aerial matching.

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', 'kornia', '-q'], check=True)

REPO = os.path.dirname(os.path.abspath('__file__'))
DATA = os.path.join(REPO, 'UAV_VisLoc_example')
OUT  = os.path.join(REPO, 'results')
os.makedirs(OUT, exist_ok=True)
sys.modules.pop('loftr_pipeline', None)
sys.modules.pop('visloc_utils', None)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import loftr_pipeline as pl

pl.BASE      = f"{DATA}/03"
pl.SAT_TIF   = f"{pl.BASE}/satellite03.tif"
pl.DRONE_DIR = f"{pl.BASE}/drone"
pl.DRONE_CSV = f"{pl.BASE}/03.csv"
pl.SAT_CSV   = f"{DATA}/satellite_ coordinates_range.csv"
pl.OUT_CSV   = f"{OUT}/loftr_results.csv"
pl.VIZ_DIR   = f"{OUT}/loftr_viz"

sys.argv = ['', '--limit', '100', '--pretrained', 'outdoor',
            '--conf', '0.0', '--dist', '25', '--visualize']
pl.main()

---
## Section 4 — RoMa

**RoMa** (Robust Dense Feature Matching) produces a dense warp field between image pairs using a DINOv2 backbone, then samples correspondence points from it. No keypoint detection step. Typically the most accurate dense matcher for large viewpoint and scale changes. GPU required; slowest of all methods.

Pretrained on MegaDepth (`outdoor`) — correct for aerial/satellite scenes.

### 4a — RoMa (outdoor)

Dense warp-based matching with DINOv2 backbone. Samples 5000 correspondences from the predicted warp field.

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, '-m', 'pip', 'install', 'romatch', '-q'], check=True)

REPO = os.path.dirname(os.path.abspath('__file__'))
DATA = os.path.join(REPO, 'UAV_VisLoc_example')
OUT  = os.path.join(REPO, 'results')
os.makedirs(OUT, exist_ok=True)
sys.modules.pop('roma_pipeline', None)
sys.modules.pop('visloc_utils', None)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import roma_pipeline as pl

pl.BASE      = f"{DATA}/03"
pl.SAT_TIF   = f"{pl.BASE}/satellite03.tif"
pl.DRONE_DIR = f"{pl.BASE}/drone"
pl.DRONE_CSV = f"{pl.BASE}/03.csv"
pl.SAT_CSV   = f"{DATA}/satellite_ coordinates_range.csv"
pl.OUT_CSV   = f"{OUT}/roma_results.csv"
pl.VIZ_DIR   = f"{OUT}/roma_viz"

sys.argv = ['', '--limit', '100', '--pretrained', 'outdoor',
            '--conf', '0.0', '--num-matches', '5000', '--dist', '25', '--visualize']
pl.main()